In [ ]:
#Cell-0
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from sklearn.ensemble import IsolationForest


In [ ]:
#Cell-1
#Dataset paths
BASE_DIR = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "training_videos")
TEST_DIR  = os.path.join(BASE_DIR, "testing_videos")

#Device selection
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
#Cell-2
#Load ResNet18 (no pretrained weights)
resnet = models.resnet18(weights=None)

#Remove classification layer
resnet.fc = nn.Identity()

#Freeze all parameters since we are not training the network
for param in resnet.parameters():
    param.requires_grad = False

#Set model to evaluation mode
resnet.eval().to(device)


In [ ]:
#Cell-3
#Image preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((32, 32)),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
#Cell-4
#CNN feature extraction
@torch.no_grad()
def extract_cnn_feature(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    x = transform(img).unsqueeze(0).to(device)
    feat = resnet(x)

    return feat.cpu().numpy().flatten()


In [ ]:
#Cell-5
#Extract training features
X_train = []

for vid in sorted(os.listdir(TRAIN_DIR)):
    frames = sorted(os.listdir(os.path.join(TRAIN_DIR, vid)))
    for f in frames:
        path = os.path.join(TRAIN_DIR, vid, f)
        X_train.append(extract_cnn_feature(path))

X_train = np.array(X_train)
print("Train features:", X_train.shape)


In [ ]:
#Cell-6
#Train Isolation Forest
iso = IsolationForest(
    n_estimators=300,
    contamination=0.10,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train)


In [ ]:
#Cell-7
#Inference on test data
ids, scores = [], []

for vid in sorted(os.listdir(TEST_DIR)):
    frames = sorted(os.listdir(os.path.join(TEST_DIR, vid)))
    vid_scores = []

    for f in frames:
        path = os.path.join(TEST_DIR, vid, f)
        feat = extract_cnn_feature(path)
        score = -iso.decision_function([feat])[0]
        vid_scores.append(score)

    #Temporal smoothing per video
    vid_scores = (
        pd.Series(vid_scores)
        .rolling(9, center=True, min_periods=1)
        .mean()
    )

    for f, s in zip(frames, vid_scores):
        ids.append(f"{vid}_{f}")
        scores.append(s)


In [ ]:
#Cell-8
#Normalize scores
scores = np.array(scores)
scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)# 1e-8 is added to the denominator to avoid division by zero


In [ ]:
#Cell-9
#Save submission
submission = pd.DataFrame({
    "Id": ids,
    "Predicted": scores
})

submission.to_csv("submission.csv", index=False)
